This notebook was developped to run multiple scenarios and experiments by generating the fitting config file and running the synth task. A try / except condition can be added to test multiple scenarios / experiments automatically.

Before running it, the following conditions must be met:
- the cmip6 climatic data from the scenarios / experiments to run must be loaded in the data/input/cmip6_data folder. This can be done with the notebook prepare_cmip6_data.ipynb, or directly by running the pangeo task.
- the natural earth datasets with coastline and land shapes must be loaded in the data/input folder. To do so, go to https://www.quickmaptools.com/download-natural-earth and dowload coastline and land data by selecting the scale 1:10m and Shapefile (ZIP) data.
- the file used for bias correction (named ERA5_benchmark_100tracks_bias_corrections.csv), computed on ERA5 data, must be aded in data/input/bias_correction.
- historical cyclone tracks data (from NOAA) must be downloaded from the following link : https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r01/access/csv/. The file must be named ibtracs.since1980.list.v04r01.csv and placed in data/input.
- Catherina fit coefficients must be placed in the folder input/fit.

In [5]:
import toml
import subprocess
import time

Definition of scenarios and experiments to run

In [6]:
# scenarios = {
#     'ACCESS-CM2':[["historical"], ["ssp245"], ["ssp370"], ["ssp585"]],
#     'FGOALS-g3':[["historical"], ["ssp370"], ["ssp585"]],
#     'IPSL-CM6A-LR':[["historical"], ["ssp370"], ["ssp585"]],
#     'MPI-ESM1-2-LR':[["historical"], ["ssp370"], ["ssp585"]],
# }

scenarios = {
    'ACCESS-CM2':[["historical"]],
}

In [7]:
def get_dates(experiment):
    # Retrieve the start and end years for the given experiment, depending on if it's historical or future experiment
    if experiment == "historical":
        start_year = 1980
        end_year = 2015
    else:
        start_year = 2025
        end_year = 2100
    return start_year, end_year

Main loop

In [ ]:
# The total number of seeds run is equel to: number_times_running_seeds * number_seeds_per_run
number_times_running_seeds = 1
number_seeds_per_run = 1

time_dict = {}

for model, experiments in scenarios.items():

    time_dict.setdefault(model, {})

    for experiment in experiments:
        print(experiment)

        ## Generate config file ##

        start_year, end_year = get_dates(experiment[0])

        with open("config_model.toml", "r") as f:
            config = toml.load(f)

        config["main_params"]["models"] = [model]
        config["main_params"]["experiments"] = experiment
        config["main_params"]["start_year"] = start_year
        config["main_params"]["end_year"] = end_year
        config["main_params"]["n_seeds"] = number_seeds_per_run

        with open("config.toml", "w") as f:
            toml.dump(config, f)
        

        ## Run experimnent ##

        print(f"Running with: {model} {experiment[0]}" )

        time_dict[model][experiment[0]] = []

        for i in range(number_times_running_seeds):
            print(f" - Running seeds: {i}")
            start_time = time.perf_counter()

            result = subprocess.run(
                ["pixi", "run", "synth"],
                capture_output=True,
                text=True
            )

            print("STDOUT:\n", result.stdout)
            print("STDERR:\n", result.stderr)
            print("RETURN CODE:", result.returncode)

            elapsed = time.perf_counter() - start_time
            print(f"--- {elapsed:.2f} seconds ---")
            time_dict[model][experiment[0]].append(elapsed)

['historical']
Running with: ACCESS-CM2 historical
 - Running seeds: 0
STDOUT:
 [06/25/26 10:42:33] INFO     Running Catherina module. | synthetic_tracks.py |  
                             line 39                                            
Starting at seed number:  0
--- Correcting climate bias ---
intensifying tracks...
['SID', 'step', 'lat_left', 'lon_left', 'datetime', 'time', 'y', 'x', 'nshr', 'MSLP', 'T_strat', 'SST', 'height', 'lat_right', 'lon_right', 'distance_track_env', 'thermo_eff', 'seed', 'year', 'month', 'geometry']
['basin', 'geometry']

===== STOP STEP DEBUG =====
Number of stopped storms: 1879
First few stop_step entries:
[((0, 2625717191), 1), ((0, 2625716192), 3), ((0, 2625716779), 3), ((0, 2625715680), 4), ((0, 2625715856), 4), ((0, 2625716137), 4), ((0, 2625716148), 4), ((0, 2625716421), 4), ((0, 2625716813), 4), ((0, 2625716851), 4)]
Rows before filtering: 260790
Rows removed: 160197
Rows kept: 100593

Sample removed rows:
    seed         SID  step
0      0  26